#  Embedding analysis of axis ratio, effective radius, and their relationship in AstroDINO embedding space

This note book will build a dataset that includes both axis ratio (axis ratio) and effective radius (effective radius) labels, compute AstroDINO embeddings, and visualize the distribution of these two labels in PCA, t-SNE, and UMAP reduced spaces.

- Reference the dataset writing methods in linear_probe_axis_ratio and linear_probe_re
- Support error threshold filtering for axis ratio and radius
- Study the performance of these two physical quantities in embedding space

In [ ]:
import os
import sys
import glob
import numpy as np
import h5py
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from omegaconf import OmegaConf
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

PROJECT_ROOT = "/u/yacheng/projects/ssl_outthere"
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "encoder_image/astrodino/benchmark/linearprobe"))

from dinov2.eval.setup import build_model_for_eval
from encoder_image.astrodino.train.data.augmentations import ToRGB

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

PIXEL_SCALE_MAS = 30  # mas per pixel
DEG_TO_PIXEL = 3600 * 1000 / PIXEL_SCALE_MAS  # = 120000 pixels per degree

In [ ]:
# Configuration parameters
MODEL_CONFIG = "/u/yacheng/projects/ssl_outthere/encoder_image/astrodino/model/astrodino_f150w_vitb/config.yaml"
MODEL_WEIGHTS = "/u/yacheng/projects/ssl_outthere/encoder_image/astrodino/model/astrodino_f150w_vitb/eval/training_149999/teacher_checkpoint.pth"
DATA_ROOT = "/u/yacheng/projects/ssl_outthere/images/jwst/f150w"

BATCH_SIZE = 64
MAX_SAMPLES = 10000  # -1 means all samples
SEED = 42

# Error filtering thresholds
AXIS_RATIO_REL_ERR_THRESH = 1  # Relative axis ratio error threshold
RADIUS_SERSIC_REL_ERR_THRESH = 1  # Relative effective radius error threshold
MORPH_FLAG_ERR_THRESH = 1

In [ ]:
class JWSTAxisRatioRadiusDataset(Dataset):
    """
    Load the JWST dataset that provides both axis ratio and effective radius.
    Targets: log10(radius_sersic_pixels), axis_ratio, morphology flag
    Supports error filtering.
    """
    def __init__(self, root, crop_size=64, max_samples=-1, seed=42,
                 axis_ratio_rel_err_thresh=0.1, radius_sersic_rel_err_thresh=0.1, morph_flag_err_thresh=0.1,
                 effective_radius_min=2.5):
        self.crop_size = crop_size
        self.to_rgb = ToRGB()
        self.center_crop = transforms.CenterCrop(crop_size)
        self.rng = np.random.default_rng(seed=seed)
        self.axis_ratio_rel_err_thresh = axis_ratio_rel_err_thresh
        self.radius_sersic_rel_err_thresh = radius_sersic_rel_err_thresh
        self.morph_flag_err_thresh=morph_flag_err_thresh, 
        self.effective_radius_min = effective_radius_min
        
        self._files = []
        h5_files = sorted(glob.glob(os.path.join(root, "*.h5")))
        for fpath in h5_files:
            try:
                f = h5py.File(fpath, 'r')
                if 'radius_sersic' in f and 'axis_ratio' in f:
                    self._files.append(f)
            except Exception as e:
                print(f"Error: {e}")
        print(f"Loaded {len(self._files)} files with both radius_sersic and axis_ratio")

        self._valid_indices = []  # (file_idx, local_idx, log10_radius_pix, axis_ratio, morph_flag)
        for file_idx, f in enumerate(tqdm(self._files, desc="Indexing")):
            rs = f['radius_sersic'][:]
            ar = f['axis_ratio'][:]
            morph_flag = f['morph_flag_f150w'][:]
            morph_flag_err = f['morph_flag_f150w'][:]
            # error fields
            rs_err = f['radius_sersic_err'][:]
            ar_err = f['axis_ratio_err'][:]
            # Validity mask
            valid_mask = np.isfinite(rs) & (rs > 0) & np.isfinite(ar) & (ar > 0) & (ar <= 1)
            if rs_err is not None:
                valid_mask &= np.isfinite(rs_err)
                if self.radius_sersic_rel_err_thresh is not None:
                    rel_err = np.where(rs > 0, rs_err / rs, np.inf)
                    valid_mask &= (rel_err <= self.radius_sersic_rel_err_thresh)
            if ar_err is not None:
                valid_mask &= np.isfinite(ar_err)
                if self.axis_ratio_rel_err_thresh is not None:
                    rel_err = np.where(ar > 0, ar_err / ar, np.inf)
                    valid_mask &= (rel_err <= self.axis_ratio_rel_err_thresh)
            if morph_flag is not None:
                valid_mask &= np.isfinite(morph_flag)
                valid_mask &= (morph_flag >= 0) & (morph_flag <= 3)
                valid_mask &= (morph_flag != 2)
            if self.effective_radius_min is not None:
                if 'radius_sersic' not in f:
                    valid_mask[:] = False
                else:
                    re = f['radius_sersic'][:]
                    re_pix = re * DEG_TO_PIXEL
                    valid_mask &= np.isfinite(re_pix)
                    valid_mask &= (re_pix >= self.effective_radius_min)
            if morph_flag_err is not None:
                valid_mask &= np.isfinite(morph_flag_err)
                valid_mask &= (morph_flag <= self.morph_flag_err_thresh)
                
            valid_local_indices = np.where(valid_mask)[0]
            for local_idx in valid_local_indices:
                radius_pix = float(rs[local_idx]) * DEG_TO_PIXEL
                log_r = np.log10(radius_pix)
                axis_ratio = float(ar[local_idx])
                morph_label = float(morph_flag[local_idx])
            
                # Constrain log_r range
                if log_r >= 0 and log_r < 2 and axis_ratio > 0.05 and axis_ratio < 0.95:
                    self._valid_indices.append((file_idx, local_idx, log_r, axis_ratio, morph_label))
                               
        print(f"Total valid samples: {len(self._valid_indices)}")
        if max_samples > 0 and max_samples < len(self._valid_indices):
            indices = self.rng.choice(len(self._valid_indices), size=max_samples, replace=False)
            self._valid_indices = [self._valid_indices[i] for i in indices]
            print(f"Subsampled to {len(self._valid_indices)} samples")

    def __len__(self):
        return len(self._valid_indices)

    def __getitem__(self, index):
        file_idx, local_idx, log_radius, axis_ratio, morph_label = self._valid_indices[index]
        img = self._files[file_idx]['image'][local_idx].astype('float32')
        img = np.repeat(img[np.newaxis, :, :], 3, axis=0)
        tensor = torch.from_numpy(img)
        tensor = self.center_crop(tensor)
        tensor = torch.from_numpy(self.to_rgb(tensor.numpy()))
        return tensor, torch.tensor([log_radius, axis_ratio, morph_label], dtype=torch.float32)

In [ ]:
# Load model
print("Loading model...")
cfg = OmegaConf.load(MODEL_CONFIG)
model = build_model_for_eval(cfg, pretrained_weights=MODEL_WEIGHTS)
model = model.to(DEVICE)
model.eval()
print(f"Model loaded, crop_size={cfg.crops.global_crops_size}")

# Build dataset
print("Building dataset...")
dataset = JWSTAxisRatioRadiusDataset(
    DATA_ROOT,
    crop_size=cfg.crops.global_crops_size,
    max_samples=MAX_SAMPLES,
    seed=SEED,
    axis_ratio_rel_err_thresh=AXIS_RATIO_REL_ERR_THRESH,
    radius_sersic_rel_err_thresh=RADIUS_SERSIC_REL_ERR_THRESH,
    morph_flag_err_thresh=MORPH_FLAG_ERR_THRESH,
    effective_radius_min=0.5
 )
print(f"Dataset size: {len(dataset)}")

In [ ]:
# Compute label distributions
labels = np.array([pair[2:4] for pair in dataset._valid_indices])
log_radii = labels[:, 0]
axis_ratios = labels[:, 1]
morph_labels = np.array([pair[4] for pair in dataset._valid_indices])
valid_morph_mask = np.isfinite(morph_labels)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
ax = axes[0]
ax.hist(log_radii, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax.set_xlabel('log₁₀(r_eff [pixels])')
ax.set_ylabel('Count')
ax.set_title('log₁₀(Effective Radius) distribution')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.hist(axis_ratios, bins=50, edgecolor='black', alpha=0.7, color='orange')
ax.set_xlabel('Axis Ratio (b/a)')
ax.set_ylabel('Count')
ax.set_title('Axis Ratio distribution')
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.hist(morph_labels)

plt.tight_layout()
plt.show()

print(f"log₁₀(r_eff) stats: mean={log_radii.mean():.3f}, median={np.median(log_radii):.3f}, std={log_radii.std():.3f}")
print(f"axis_ratio stats: mean={axis_ratios.mean():.3f}, median={np.median(axis_ratios):.3f}, std={axis_ratios.std():.3f}")
if valid_morph_mask.any():
    unique, counts = np.unique(morph_labels[valid_morph_mask].astype(int), return_counts=True)
    morph_summary = ', '.join([f"{int(u)}:{c}" for u, c in zip(unique, counts)])
    print(f"Morphology label counts: {morph_summary}")
else:
    print("Morphology labels are missing for these samples.")

In [ ]:
# Compute embeddings
print("Computing embeddings...")
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)
all_embeddings = []
all_labels = []
with torch.no_grad():
    for batch_imgs, batch_labels in tqdm(dataloader, desc="Embedding batches"):
        batch_imgs = batch_imgs.to(DEVICE)
        emb = model(batch_imgs)
        if isinstance(emb, tuple):
            emb = emb[0]
        if emb.dim() > 2:
            emb = emb.view(emb.size(0), -1)
        all_embeddings.append(emb.cpu().numpy())
        all_labels.append(batch_labels.numpy())
embeddings = np.concatenate(all_embeddings, axis=0)
labels = np.concatenate(all_labels, axis=0)
print(f"Embeddings shape: {embeddings.shape}")
print(f"Labels shape: {labels.shape}")

In [ ]:
# Dimensionality reduction visualization
n_vis = min(10000, len(embeddings))
vis_idx = np.random.default_rng(SEED).choice(len(embeddings), size=n_vis, replace=False)
emb_vis = embeddings[vis_idx]
log_r_vis = labels[vis_idx, 0]
axis_ratio_vis = labels[vis_idx, 1]
morph_labels = np.array([entry[4] for entry in dataset._valid_indices])
morph_vis = morph_labels[vis_idx]

print(f"Visualization samples: {n_vis}")

# PCA
top_n_pca = 50
print("Running PCA...")
pca = PCA(n_components=top_n_pca)
emb_pca = pca.fit_transform(emb_vis)
print(f"Explained variance (top {top_n_pca}): {pca.explained_variance_ratio_.sum():.2%}")

# t-SNE
print("Running t-SNE...")
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, max_iter=1000, verbose=1)
emb_tsne = tsne.fit_transform(emb_pca)

# UMAP
print("Running UMAP...")
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED, verbose=True)
emb_umap = reducer.fit_transform(emb_pca)
print("UMAP done.")

In [ ]:
# PCA/t-SNE/UMAP colored by log_r, axis_ratio, and morphology
MORPH_LABEL_NAMES = {
    0: 'Spheroid',
    1: 'Disk-dominated',
    2: 'Irregular',
    3: 'Bulge-dominated',
}
MORPH_LABEL_COLORS = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']
morph_cmap = ListedColormap(MORPH_LABEL_COLORS)
morph_norm = BoundaryNorm(np.arange(-0.5, len(MORPH_LABEL_NAMES) + 0.5, 1), morph_cmap.N)

fig, axes = plt.subplots(3, 3, figsize=(18, 16))
dim_titles = ['PCA', 't-SNE', 'UMAP']
dim_embeddings = [emb_pca, emb_tsne, emb_umap]
color_configs = [
    ('log₁₀(r_eff)', log_r_vis, 'viridis', 'log₁₀(r_eff [pixels])', False),
    ('Axis Ratio', axis_ratio_vis, 'plasma', 'Axis Ratio (b/a)', False),
    ('Morphology', morph_vis, morph_cmap, 'Morphology class', True),
]

for row_idx, (title, embedding) in enumerate(zip(dim_titles, dim_embeddings)):
    for col_idx, (label, values, cmap, colorbar_label, is_discrete) in enumerate(color_configs):
        ax = axes[row_idx, col_idx]
        ax.scatter(embedding[:, 0], embedding[:, 1], c='lightgray', s=6, alpha=0.2)
        if is_discrete:
            mask = np.isfinite(values)
            if mask.any():
                sc = ax.scatter(embedding[mask, 0], embedding[mask, 1], c=values[mask], cmap=cmap, norm=morph_norm, s=6, alpha=0.4)
                cb = plt.colorbar(sc, ax=ax, ticks=list(MORPH_LABEL_NAMES.keys()), label=colorbar_label, fraction=0.046, pad=0.04)
                cb.ax.set_yticklabels([MORPH_LABEL_NAMES[k] for k in MORPH_LABEL_NAMES.keys()])
            else:
                ax.text(0.5, 0.5, 'No morphology labels', ha='center', va='center', transform=ax.transAxes, fontsize=10)
        else:
            sc = ax.scatter(embedding[:, 0], embedding[:, 1], c=values, cmap=cmap, s=6, alpha=0.4)
            plt.colorbar(sc, ax=ax, label=colorbar_label, fraction=0.046, pad=0.04)
        ax.set_title(f'{title} colored by {label}')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# Animation: show only points within the current interval per frame; higher frame rate, blank start, top row for axis ratio and bottom row for r_eff

import matplotlib.pyplot as plt

import matplotlib.animation as animation

from matplotlib.colors import Normalize

import numpy as np

# Frame count and frame rate

n_frames = 70

interval_ms = 80  # Frame interval; smaller values speed up playback

# Top row: axis ratio, bottom row: r_eff

row_vars = [axis_ratio_vis, log_r_vis]

row_labels = ['Axis Ratio (b/a)', 'log₁₀(r_eff [pixels])']

row_cmaps = ['plasma', 'viridis']

row_bins = [np.linspace(np.min(v), np.max(v), n_frames+1) for v in row_vars]

embeddings_list = [emb_pca, emb_tsne, emb_umap]

emb_titles = ['PCA', 't-SNE', 'UMAP']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

scs = [[None for _ in range(3)] for _ in range(2)]

norms = [Normalize(vmin=np.min(v), vmax=np.max(v)) for v in row_vars]

# Initialize with light gray background for all points; each frame will highlight the current interval

for row in range(2):

    for col in range(3):

        axes[row, col].scatter(embeddings_list[col][:, 0], embeddings_list[col][:, 1], c='lightgray', s=8, alpha=0.2)

        sc = axes[row, col].scatter([], [], c=[], cmap=row_cmaps[row], s=16, alpha=0.85, norm=norms[row])

        axes[row, col].set_title(f'{emb_titles[col]}')

        axes[row, col].set_xticks([])

        axes[row, col].set_yticks([])

        axes[row, col].grid(False)

        scs[row][col] = sc

    axes[row, 0].set_ylabel(row_labels[row], fontsize=14)

fig.tight_layout()

def update(frame):

    for row in range(2):

        mask = (row_vars[row] >= row_bins[row][frame]) & (row_vars[row] < row_bins[row][frame+1])

        for col in range(3):

            x = embeddings_list[col][mask, 0]

            y = embeddings_list[col][mask, 1]

            c = row_vars[row][mask]

            scs[row][col].set_offsets(np.c_[x, y])

            if len(c) > 0:

                scs[row][col].set_array(c)

            else:

                scs[row][col].set_array(np.array([]))

    fig.suptitle(f'Frame {frame+1}/{n_frames}\nAxis Ratio: {row_bins[0][frame]:.2f}~{row_bins[0][frame+1]:.2f} | log₁₀(r_eff): {row_bins[1][frame]:.2f}~{row_bins[1][frame+1]:.2f}', fontsize=16)

    return [sc for row in scs for sc in row]

ani = animation.FuncAnimation(fig, update, frames=n_frames, blit=False, interval=interval_ms)

for row in range(2):

    plt.colorbar(scs[row][0], ax=axes[row, :], orientation='vertical', fraction=0.02, pad=0.04, label=row_labels[row])

plt.show()

ani.save('embedding_axisratio_reff_separate.gif', writer='pillow', fps=1000//interval_ms)

In [ ]:
# Animation: step through r_eff intervals per frame while coloring points by axis ratio
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import Normalize
import numpy as np

n_frames = 70  # Frame count with finer r_eff bins
interval_ms = 80  # Time per frame
repeat_pause = 1  # Increase to slow down transitions

# r_eff intervals
re_bins = np.linspace(np.min(log_r_vis), np.max(log_r_vis), n_frames+1)
embeddings_list = [emb_pca, emb_tsne, emb_umap]
emb_titles = ['PCA', 't-SNE', 'UMAP']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
scs = [None for _ in range(3)]
norm = Normalize(vmin=np.min(axis_ratio_vis), vmax=np.max(axis_ratio_vis))

# Initialize with light gray background for all points; frames will highlight the current r_eff band
for col in range(3):
    axes[col].scatter(embeddings_list[col][:, 0], embeddings_list[col][:, 1], c='lightgray', s=8, alpha=0.2)
    sc = axes[col].scatter([], [], c=[], cmap='plasma', s=16, alpha=0.85, norm=norm)
    axes[col].set_title(f'{emb_titles[col]}')
    axes[col].set_xticks([])
    axes[col].set_yticks([])
    axes[col].grid(False)
    scs[col] = sc
axes[0].set_ylabel('')
fig.tight_layout()

# Update function
def update_re(frame):
    true_frame = frame // repeat_pause
    mask = (log_r_vis >= re_bins[true_frame]) & (log_r_vis < re_bins[true_frame+1])
    for col in range(3):
        x = embeddings_list[col][mask, 0]
        y = embeddings_list[col][mask, 1]
        c = axis_ratio_vis[mask]
        scs[col].set_offsets(np.c_[x, y])
        if len(c) > 0:
            scs[col].set_array(c)
        else:
            scs[col].set_array(np.array([]))
    fig.suptitle(f'Frame {true_frame+1}/{n_frames}\nlog₁₀(r_eff): {re_bins[true_frame]:.2f}~{re_bins[true_frame+1]:.2f}', fontsize=16)
    return scs

total_frames = n_frames * repeat_pause
ani_re = animation.FuncAnimation(fig, update_re, frames=total_frames, blit=False, interval=interval_ms)
plt.colorbar(scs[0], ax=axes, orientation='vertical', fraction=0.02, pad=0.04, label='Axis Ratio (b/a)')
plt.show()
ani_re.save('embedding_re_interval_coloredby_axisratio.gif', writer='pillow', fps=1000//interval_ms)
